Install dependencies from the root requirements-lock.txt before running these tests.

In [1]:

import numpy as np
import pytest
import ipytest
from pathlib import Path
import nbformat
import contextlib
import io

ipytest.autoconfig()

# Đọc hàm Gauss_Seidel từ file .ipynb
def load_gauss_seidel(filename):
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {filename}")

    notebook = nbformat.read(path, as_version=4)
    namespace = {"np": np}

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        source = cell.source

        # Chỉ lấy phần định nghĩa hàm, không chạy ví dụ
        if "def Gauss_Seidel(" in source:
            source = source.split("# Ví dụ")[0]

            with contextlib.redirect_stdout(io.StringIO()):
                exec(source, namespace)

            return namespace["Gauss_Seidel"]

    raise ValueError(f"Không tìm thấy hàm Gauss_Seidel trong {filename}")


# Nạp hai phiên bản thuật toán
gauss_seidel_v1 = load_gauss_seidel("gauss_seidel.ipynb")
gauss_seidel_v2 = load_gauss_seidel("gauss_seidel_v2.ipynb")


# TEST 1: Ví dụ trong bài giảng
@pytest.mark.parametrize(
    "solver",
    [gauss_seidel_v1, gauss_seidel_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"]
)
def test_vi_du_trong_bai(solver):

    if solver is gauss_seidel_v1:
        A = np.array([
            [9, 5, 2],
            [1, 7, 1],
            [-3, -1, -7]
        ], dtype=float)

        b = np.array([1, 6, -26], dtype=float)

    else:
        A = np.array([
            [5, -1, 2],
            [1, 4, -1],
            [1, 1, 4]
        ], dtype=float)

        b = np.array([8, -4, 4], dtype=float)

    result = solver(A, b, max_blap=100, esl=1e-8)

    assert result is not None

    np.testing.assert_allclose(
        A @ result,
        b,
        atol=1e-5,
        rtol=0
    )


# TEST 2: Ví dụ tự chọn có nghiệm đúng
@pytest.mark.parametrize(
    "solver",
    [gauss_seidel_v1, gauss_seidel_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"]
)
def test_nghiem_dung_tu_chon(solver):

    A = np.array([
        [4, 1],
        [1, 3]
    ], dtype=float)

    b = np.array([6, 7], dtype=float)

    expected = np.array([1, 2], dtype=float)

    result = solver(A, b, max_blap=100, esl=1e-8)

    assert result is not None

    np.testing.assert_allclose(
        result,
        expected,
        atol=1e-5,
        rtol=0
    )


# TEST 3: Trường hợp biên - không thỏa q < 1
@pytest.mark.parametrize(
    "solver",
    [gauss_seidel_v1, gauss_seidel_v2],
    ids=["Dung_so_buoc_lap", "Dung_theo_sai_so"]
)
def test_khong_thoa_dieu_kien_q(solver):

    A = np.array([
        [1, 2],
        [2, 1]
    ], dtype=float)

    b = np.array([3, 3], dtype=float)

    result = solver(A, b, max_blap=10, esl=1e-6)

    assert result is None


# Chạy toàn bộ 6 ca kiểm thử
ipytest.run("-v")

======================================= test session starts =======================================
platform win32 -- Python 3.12.7, pytest-7.4.4, pluggy-1.0.0
rootdir: c:\Users\ADMIN\PPS_T2_cu\Gau-Jacobi
plugins: anyio-4.2.0, typeguard-4.3.0
collected 6 items

t_3c36f8a6082747939768471febd07f4e.py ......                                                 [100%]

======================================== 6 passed in 0.07s ========================================


<ExitCode.OK: 0>